# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all entities via their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
metadata = dataset.metadata  # Not a dict -- do not subscript

# Print out the dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset uses Croissant schema, where entities are uniquely referenced via their `@id` field. Let's enumerate all record sets and their fields by `@id`.

In [ ]:
# List available record sets and their fields by @id
record_sets = dataset.metadata.record_sets  # This returns a list of RecordSet objects

print("Available RecordSets:")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}")
    print(f"    Name: {rs.name}")
    print(f"    Description: {getattr(rs, 'description', None)}")
    if hasattr(rs, 'fields'):
        print("    Fields:")
        for f in rs.fields:
            print(f"      Field @id: {f.id}, name: {f.name}, dataType: {getattr(f, 'data_type', None)}")
    else:
        print("    No fields listed.")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** All data is referenced by its `@id`. DataFrames for each record set are created and indexed by the corresponding `@id`.

In [ ]:
# Extract data from each record set using @id
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load records for the given record set
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"DataFrame for {record_set_id}: {dataframes[record_set_id].shape[0]} rows, {dataframes[record_set_id].shape[1]} columns")

# Example: examine the first record set and its columns
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns in {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll reference fields by their `@id`. For demonstration, let's select a numeric field and perform filtering and normalization. We'll also group by a categorical field if present.

In [ ]:
# Choose a record set
rs_idx = 0  # You may choose another index as needed
record_set_id = record_set_ids[rs_idx]
df = dataframes[record_set_id]

# List numeric fields by @id
numeric_fields = []
if hasattr(record_sets[rs_idx], 'fields'):
    for f in record_sets[rs_idx].fields:
        if getattr(f, 'data_type', None) in ['schema:Float', 'schema:Integer', 'Float', 'Integer', 'Number']:
            numeric_fields.append(f.id)

print(f"Numeric fields in {record_set_id}: {numeric_fields}")

# Select the first numeric field for analysis
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    # If the column exists in DataFrame
    if numeric_field_id in df.columns:
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        # Filter records
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field
        group_field_id = None
        # Find a field with datatype 'schema:Text' or 'Text' and not the numeric_field
        for f in record_sets[rs_idx].fields:
            if getattr(f, 'data_type', None) in ['schema:Text', 'Text'] and f.id != numeric_field_id:
                if f.id in df.columns:
                    group_field_id = f.id
                    break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print(f"Numeric field {numeric_field_id} not found in DataFrame columns: {df.columns.tolist()}")
else:
    print(f"No numeric fields found in {record_set_id}.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, referencing axes by `@id`.

Below, we'll plot the distribution of the selected numeric field and its normalized values, and if a grouping field exists, a bar plot of grouped means.

In [ ]:
# Visualization
if numeric_fields and numeric_field_id in df.columns:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(df[numeric_field_id].dropna(), bins=20, color='skyblue', edgecolor='black')
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    ax[0].set_xlabel(numeric_field_id)
    ax[0].set_ylabel("Frequency")

    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        ax[1].hist(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=20, color='coral', edgecolor='black')
        ax[1].set_title(f"Normalized {numeric_field_id} (filtered)")
        ax[1].set_xlabel(f"{numeric_field_id}_normalized")
        ax[1].set_ylabel("Frequency")

    plt.tight_layout()
    plt.show()

    # If grouped data exists
    if group_field_id and 'grouped_df' in locals():
        grouped_df.plot(kind='bar', figsize=(8, 4))
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to load, explore, filter, transform, and visualize structured data referenced via Croissant `@id`s using the `mlcroissant` library.
- We referenced record sets, fields, and columns using their `@id` exclusively.
- For further analysis, consult the Croissant schema for additional metadata and interpretation of field meanings.

**Next steps**: Apply advanced modeling, enrich EDA with contextual variable selections, and share reproducible results via Croissant-compliant workflows.